This code is a python scanner which will run in jupyter. It introduces some interesting new concepts and modules

(a) asyncio
(b) bleak

Please have a look at the code and lets see what they mean


In [2]:
import asyncio
from bleak import BleakScanner


async def scan():
    print("Scanning for BLE devices...\n")

    devices = await BleakScanner.discover(
        timeout=10,
        return_adv=True
    )

    for identifier, (device, advertisement) in devices.items():
        name = advertisement.local_name or device.name or "Unknown"

        print("Name:", name)
        print("Identifier:", device.address)
        print("Signal strength:", advertisement.rssi, "dBm")
        print("Service UUIDs:", advertisement.service_uuids)
        print("-" * 50)


await scan()
##asyncio.run(scan()) - this is for running outside jupyter

Scanning for BLE devices...

Name: [TV] Juber's TV
Identifier: 70C6C16B-EE7C-1525-3CF3-1DAD117AE0B8
Signal strength: -89 dBm
Service UUIDs: []
--------------------------------------------------
Name: [TV] Samsung 7 Series (50)
Identifier: 62D8D19C-E809-A6B9-7808-7DA97B76D828
Signal strength: -95 dBm
Service UUIDs: []
--------------------------------------------------
Name: Unknown
Identifier: 134B4DB4-ECDB-2239-8487-AFF208A22BD0
Signal strength: -74 dBm
Service UUIDs: []
--------------------------------------------------
Name: LE_WH-1000XM6
Identifier: 2B81C142-E26B-A462-880D-45CF8F20A539
Signal strength: -76 dBm
Service UUIDs: ['0000fd2a-0000-1000-8000-00805f9b34fb']
--------------------------------------------------
Name: Unknown
Identifier: CEE3608D-A2C0-888E-6322-85C86D6BAD63
Signal strength: -89 dBm
Service UUIDs: []
--------------------------------------------------
Name: Unknown
Identifier: DED3B655-DB5C-D356-3849-246956818C7D
Signal strength: -65 dBm
Service UUIDs: []
---------

The following programme is similar to the one we will be using to detect information from arduino

In [3]:
import asyncio
from bleak import BleakClient, BleakScanner


SERVICE_UUID = "19B10000-E8F2-537E-4F6C-D104768A1214"
SENSOR_UUID = "19B10001-E8F2-537E-4F6C-D104768A1214"
LED_UUID = "19B10002-E8F2-537E-4F6C-D104768A1214"


def is_nano(device, advertisement):
    """Return True when the Arduino service is advertised."""
    service_uuids = advertisement.service_uuids or []

    return SERVICE_UUID.lower() in [
        uuid.lower() for uuid in service_uuids
    ]


def sensor_notification(characteristic, data):
    """Process a measurement sent by the Arduino."""
    value = int.from_bytes(
        data,
        byteorder="little",
        signed=False
    )

    print("Sensor measurement:", value)


async def main():
    print("Looking for the Arduino Nano...")

    nano = await BleakScanner.find_device_by_filter(
        is_nano,
        timeout=20
    )

    if nano is None:
        print("Arduino not found.")
        print("Check that it is powered and advertising.")
        return

    print("Found:", nano.name)

    async with BleakClient(nano) as client:
        print("Connected:", client.is_connected)

        # Read the current sensor value
        data = await client.read_gatt_char(SENSOR_UUID)

        value = int.from_bytes(
            data,
            byteorder="little",
            signed=False
        )

        print("Current sensor value:", value)

        # Subscribe to new measurements
        await client.start_notify(
            SENSOR_UUID,
            sensor_notification
        )

        print("Receiving measurements for 30 seconds...")
        await asyncio.sleep(30)

        await client.stop_notify(SENSOR_UUID)

await main()
##asyncio.run(main())

Looking for the Arduino Nano...
Arduino not found.
Check that it is powered and advertising.
